# FRED Economic Indicators for Austin Real Estate Market

This notebook fetches economic data from the Federal Reserve Economic Data (FRED) API and generates 7 charts:

1. **Austin Employment - Office Sectors** (Professional, Financial, Government, Tech)
2. **Austin Employment - Industrial** (Trade & Transportation)
3. **Austin Employment - Retail** (Leisure & Hospitality)
4. **Austin vs National Tech Employment Growth** (indexed comparison)
5. **Austin vs Dallas vs National Wage Growth** (indexed hourly wages)
6. **Interest Rates** (10-Year Treasury vs 30-Year Mortgage)
7. **Inflation & PPI** (Core CPI, Rent CPI, Office PPIs, indexed comparison)

All charts use Aquila brand styling and are saved to the `charts/` directory.

## Setup and Helper Functions

In [51]:
import os
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from aquila_graphing_tools import aquila_styled_line_chart, AQUILA_COLORS, AQUILA_FONT

# Load environment variables
load_dotenv('aquila_graph.env')
fred_api_key = os.getenv('FRED_API_KEY')

if not fred_api_key:
    raise ValueError("FRED_API_KEY not found in aquila_graph.env")

print("✓ Environment loaded successfully")

✓ Environment loaded successfully


In [52]:
def fetch_fred_series(series_id, series_name=None):
    """
    Fetch FRED data and return as DataFrame with date and value columns.
    
    Parameters
    ----------
    series_id : str
        FRED series identifier
    series_name : str, optional
        Name for the value column (defaults to series_id)
    
    Returns
    -------
    pd.DataFrame
        DataFrame with 'date' and series_name columns
    """
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": fred_api_key,
        "file_type": "json"
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        observations = data.get("observations", [])
        
        if not observations:
            print(f"Warning: No data returned for series {series_id}")
            return pd.DataFrame()
        
        df = pd.DataFrame(observations)
        df['date'] = pd.to_datetime(df['date'])
        
        # Convert value to numeric, handling '.' as NaN
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        
        # Rename value column to series name
        column_name = series_name if series_name else series_id
        df = df[['date', 'value']].rename(columns={'value': column_name})
        
        # Drop NaN values
        df = df.dropna()
        
        print(f"✓ Fetched {len(df)} observations for {series_id} ({column_name})")
        return df
        
    except Exception as e:
        print(f"Error fetching series {series_id}: {str(e)}")
        return pd.DataFrame()

print("✓ Helper function defined")

✓ Helper function defined


## Chart 1: Austin Employment - Office Sectors

Tracks employment in office-related sectors:
- Professional & Business Services
- Financial Activities
- Government
- Information (Tech)

In [53]:
# Fetch employment data for office sectors
df_prof = fetch_fred_series('AUST448PBSV', 'Professional & Business Services')
df_fire = fetch_fred_series('AUST448FIRE', 'Financial Activities')
df_govt = fetch_fred_series('AUST448GOVT', 'Government')
df_info = fetch_fred_series('AUST448INFO', 'Information (Tech)')

# Merge all series on date
df_office = df_prof.merge(df_fire, on='date', how='outer') \
                   .merge(df_govt, on='date', how='outer') \
                   .merge(df_info, on='date', how='outer')

# Convert to long format for plotting
df_office_long = df_office.melt(id_vars=['date'], 
                                 var_name='Sector', 
                                 value_name='Employment (thousands)')

# Create chart
fig = aquila_styled_line_chart(
    df_office_long,
    x='date',
    y='Employment (thousands)',
    color='Sector',
    title='Austin Employment - Office Sectors',
    height=800
)

fig.update_yaxes(rangemode='tozero')

# Save chart
fig.write_html('charts/economic-indicators/austin_employment_office_sectors.html')
print("✓ Chart saved: austin_employment_office_sectors.html")

fig.show()

Error fetching series AUST448PBSV: 502 Server Error: Bad Gateway for url: https://api.stlouisfed.org/fred/series/observations?series_id=AUST448PBSV&api_key=52b7a19820d868eefb8ce1eed6272edb&file_type=json
✓ Fetched 431 observations for AUST448FIRE (Financial Activities)
✓ Fetched 431 observations for AUST448GOVT (Government)
✓ Fetched 431 observations for AUST448INFO (Information (Tech))


KeyError: 'date'

## Chart 2: Austin Employment - Industrial Sector

Tracks employment in Trade, Transportation & Utilities (industrial-related employment)

In [ ]:
# Fetch industrial employment data
df_industrial = fetch_fred_series('AUST448TRAD', 'Trade, Transportation & Utilities')

# Create chart
fig = aquila_styled_line_chart(
    df_industrial,
    x='date',
    y='Trade, Transportation & Utilities',
    title='Austin Employment - Industrial Sector',
    height=800
)

fig.update_yaxes(rangemode='tozero', title='Employment (thousands)')

# Save chart
fig.write_html('charts/economic-indicators/austin_employment_industrial.html')
print("✓ Chart saved: austin_employment_industrial.html")

fig.show()

## Chart 3: Austin Employment - Retail Sector

Tracks employment in Leisure & Hospitality (retail-related employment)

In [ ]:
# Fetch retail employment data
df_retail = fetch_fred_series('AUST448LEIH', 'Leisure & Hospitality')

# Create chart
fig = aquila_styled_line_chart(
    df_retail,
    x='date',
    y='Leisure & Hospitality',
    title='Austin Employment - Retail Sector',
    height=800
)

fig.update_yaxes(rangemode='tozero', title='Employment (thousands)')

# Save chart
fig.write_html('charts/economic-indicators/austin_employment_retail.html')
print("✓ Chart saved: austin_employment_retail.html")

fig.show()

## Chart 4: Austin vs National Tech Employment Growth

Compares tech employment growth between Austin and the nation, indexed to 100 at the earliest common date

In [ ]:
# Fetch tech employment data
df_austin_tech = fetch_fred_series('AUST448INFO', 'Austin Tech')
df_national_tech = fetch_fred_series('USINFO', 'National Tech')

# Merge on date
df_tech = df_austin_tech.merge(df_national_tech, on='date', how='inner')

# Find earliest common date and index both series to 100
if len(df_tech) > 0:
    base_austin = df_tech['Austin Tech'].iloc[0]
    base_national = df_tech['National Tech'].iloc[0]
    
    df_tech['Austin Tech (Index)'] = (df_tech['Austin Tech'] / base_austin) * 100
    df_tech['National Tech (Index)'] = (df_tech['National Tech'] / base_national) * 100
    
    # Convert to long format
    df_tech_long = df_tech[['date', 'Austin Tech (Index)', 'National Tech (Index)']].melt(
        id_vars=['date'],
        var_name='Region',
        value_name='Employment Index (Base 100)'
    )
    
    # Create chart
    fig = aquila_styled_line_chart(
        df_tech_long,
        x='date',
        y='Employment Index (Base 100)',
        color='Region',
        title='Austin vs National Tech Employment Growth',
        height=800
    )
    
    fig.update_yaxes(rangemode='tozero')
    
    # Save chart
    fig.write_html('charts/economic-indicators/austin_vs_national_tech_employment.html')
    print("✓ Chart saved: austin_vs_national_tech_employment.html")
    
    fig.show()
else:
    print("Warning: No overlapping data for tech employment comparison")

## Chart 5: Austin vs Dallas vs National Wage Growth

Compares wage growth between Austin, Dallas, and the nation, indexed to 100 at the earliest common date.
All data uses hourly wages for apples-to-apples comparison.

In [61]:
# Fetch hourly wage data for Austin, Dallas, and Nation
df_austin_wage = fetch_fred_series('SMU48124200500000003', 'Austin Hourly Wage')
df_dallas_wage = fetch_fred_series('SMU48191000500000003', 'Dallas Hourly Wage')
df_national_wage = fetch_fred_series('CES0500000003', 'National Hourly Wage')

# Merge all three on date
df_wage = df_austin_wage.merge(df_dallas_wage, on='date', how='inner')
df_wage = df_wage.merge(df_national_wage, on='date', how='inner')

# Index all three series to 100 at earliest common date
if len(df_wage) > 0:
    base_austin = df_wage['Austin Hourly Wage'].iloc[0]
    base_dallas = df_wage['Dallas Hourly Wage'].iloc[0]
    base_national = df_wage['National Hourly Wage'].iloc[0]
    
    df_wage['Austin Wage (Index)'] = (df_wage['Austin Hourly Wage'] / base_austin) * 100
    df_wage['Dallas Wage (Index)'] = (df_wage['Dallas Hourly Wage'] / base_dallas) * 100
    df_wage['National Wage (Index)'] = (df_wage['National Hourly Wage'] / base_national) * 100
    
    # Convert to long format
    df_wage_long = df_wage[['date', 'Austin Wage (Index)', 'Dallas Wage (Index)', 'National Wage (Index)']].melt(
        id_vars=['date'],
        var_name='Region',
        value_name='Wage Index (Base 100)'
    )
    
    # Create chart
    fig = aquila_styled_line_chart(
        df_wage_long,
        x='date',
        y='Wage Index (Base 100)',
        color='Region',
        title='Austin vs Dallas vs National Wage Growth',
        height=800
    )
    
    fig.update_yaxes(rangemode='tozero')
    
    # Save chart
    fig.write_html('charts/economic-indicators/austin_vs_dallas_vs_national_wage_growth.html')
    print("✓ Chart saved: austin_vs_dallas_vs_national_wage_growth.html")
    
    fig.show()
else:
    print("Warning: No overlapping data for wage comparison")

✓ Fetched 227 observations for SMU48124200500000003 (Austin Hourly Wage)
✓ Fetched 227 observations for SMU48191000500000003 (Dallas Hourly Wage)
✓ Fetched 238 observations for CES0500000003 (National Hourly Wage)
✓ Chart saved: austin_vs_dallas_vs_national_wage_growth.html


## Chart 6: Interest Rates - 10-Year Treasury vs 30-Year Mortgage

Tracks two key interest rates that impact commercial real estate financing

In [60]:
# Fetch interest rate data
df_treasury = fetch_fred_series('DGS10', '10-Year Treasury')
df_mortgage = fetch_fred_series('MORTGAGE30US', '30-Year Mortgage')

# Drop NA from each so there's a proper overlap
df_treasury = df_treasury.dropna(subset=['10-Year Treasury'])
df_mortgage = df_mortgage.dropna(subset=['30-Year Mortgage'])

# Find the latest of each starting date, to ensure true overlap
start_date = max(df_treasury['date'].min(), df_mortgage['date'].min())

# Filter both to start from the same date
df_treasury_overlap = df_treasury[df_treasury['date'] >= start_date]
df_mortgage_overlap = df_mortgage[df_mortgage['date'] >= start_date]

# Perform an inner join so only overlapping dates
df_rates = df_treasury_overlap.merge(df_mortgage_overlap, on='date', how='inner')

# Convert to long format for plotting
df_rates_long = df_rates.melt(
    id_vars=['date'],
    var_name='Rate Type',
    value_name='Interest Rate (%)'
)

# Double-check at least one value in each series after melt, especially for 30-Year Mortgage
if df_rates_long[df_rates_long['Rate Type'] == '30-Year Mortgage']['Interest Rate (%)'].notna().sum() == 0:
    print("Warning: No visible data for 30-Year Mortgage after filtering!")

# Create chart
fig = aquila_styled_line_chart(
    df_rates_long,
    x='date',
    y='Interest Rate (%)',
    color='Rate Type',
    title='Interest Rates - Treasury & Mortgage',
    height=800
)

# Save chart
fig.write_html('charts/economic-indicators/interest_rates_treasury_mortgage.html')
print("✓ Chart saved: interest_rates_treasury_mortgage.html")

fig.show()

✓ Fetched 15996 observations for DGS10 (10-Year Treasury)
✓ Fetched 2861 observations for MORTGAGE30US (30-Year Mortgage)
✓ Chart saved: interest_rates_treasury_mortgage.html


## Chart 7: Inflation & PPI - CPI and Office Construction Costs

Compares general inflation (Core CPI), rent inflation (Rent CPI), and construction/office cost indices (PPIs), all indexed to 100 at earliest common date

In [56]:
# Fetch inflation and price index data
df_core_cpi = fetch_fred_series('CPILFESL', 'Core CPI')
df_rent_cpi = fetch_fred_series('CUUR0000SEHC', 'Rent CPI')
df_ppi_new_office = fetch_fred_series('PCU236223236223', 'PPI - New Office Construction')
df_ppi_office_rent = fetch_fred_series('WPU43110101', 'PPI - Office Rent')
df_ppi_multi_construction = fetch_fred_series('WPUIP231120', 'PPI - Multifamily Construction (ex cap/labor/imports)')

# Merge all on date (inner join keeps only overlapping dates)
df_inflation = df_core_cpi.merge(df_rent_cpi, on='date', how='inner') \
    .merge(df_ppi_new_office, on='date', how='inner') \
    .merge(df_ppi_office_rent, on='date', how='inner') \
    .merge(df_ppi_multi_construction, on='date', how='inner')

# Index all series to 100 at earliest common date
if len(df_inflation) > 0:
    base_core = df_inflation['Core CPI'].iloc[0]
    base_rent = df_inflation['Rent CPI'].iloc[0]
    base_ppi_new_office = df_inflation['PPI - New Office Construction'].iloc[0]
    base_ppi_office_rent = df_inflation['PPI - Office Rent'].iloc[0]
    base_ppi_multi_construction = df_inflation['PPI - Multifamily Construction (ex cap/labor/imports)'].iloc[0]
    
    df_inflation['Core CPI (Index)'] = (df_inflation['Core CPI'] / base_core) * 100
    df_inflation['Rent CPI (Index)'] = (df_inflation['Rent CPI'] / base_rent) * 100
    df_inflation['PPI - New Office Construction (Index)'] = (df_inflation['PPI - New Office Construction'] / base_ppi_new_office) * 100
    df_inflation['PPI - Office Rent (Index)'] = (df_inflation['PPI - Office Rent'] / base_ppi_office_rent) * 100
    df_inflation['PPI - Multifamily Construction (Index)'] = (df_inflation['PPI - Multifamily Construction (ex cap/labor/imports)'] / base_ppi_multi_construction) * 100
    
    # Convert to long format
    df_inflation_long = df_inflation[
        ['date', 
         'Core CPI (Index)', 
         'Rent CPI (Index)', 
         'PPI - New Office Construction (Index)', 
         'PPI - Office Rent (Index)',
         'PPI - Multifamily Construction (Index)']
    ].melt(
        id_vars=['date'],
        var_name='Inflation Type',
        value_name='Index (Base 100)'
    )
    
    # Create chart
    fig = aquila_styled_line_chart(
        df_inflation_long,
        x='date',
        y='Index (Base 100)',
        color='Inflation Type',
        title='Inflation & PPI - Core CPI, Rent CPI, New Office, Multifamily, Office Rent (Indexed)',
        height=800
    )
    
    
    # Save chart
    fig.write_html('charts/economic-indicators/inflation_cpi_ppi_office.html')
    print("✓ Chart saved: inflation_cpi_ppi_office.html")
    
    fig.show()
else:
    print("Warning: No overlapping data for inflation/ppi comparison")

✓ Fetched 827 observations for CPILFESL (Core CPI)
✓ Fetched 516 observations for CUUR0000SEHC (Rent CPI)
✓ Fetched 234 observations for PCU236223236223 (PPI - New Office Construction)
✓ Fetched 204 observations for WPU43110101 (PPI - Office Rent)
✓ Fetched 132 observations for WPUIP231120 (PPI - Multifamily Construction (ex cap/labor/imports))
✓ Chart saved: inflation_cpi_ppi_office.html


## Summary

All 7 charts have been generated and saved to the `charts/` directory:

1. ✓ `austin_employment_office_sectors.html`
2. ✓ `austin_employment_industrial.html`
3. ✓ `austin_employment_retail.html`
4. ✓ `austin_vs_national_tech_employment.html`
5. ✓ `austin_vs_dallas_vs_national_wage_growth.html`
6. ✓ `interest_rates_treasury_mortgage.html`
7. ✓ `inflation_cpi_ppi_office.html`

These charts are ready to be published to GitHub Pages and linked in README.md.